# ColabProTrek-HM: A ColabPLM for Hard-Negative Protein-Text Retrieval

**Track B mapping.** This notebook wraps an existing protein language model system into a ColabPLM-style runnable artifact.

What this notebook does:
- Demonstrates a lightweight wrapper around ProTrek.
- Runs protein sequence ↔ natural-language function retrieval.
- Uses **ProTrek-35M** as the default model.
- Optionally compares baseline ProTrek-35M with a fine-tuned **ProTrek-HM inference-only checkpoint**.

What is not used by default:
- ProTrek_650M.
- FAISS database indexes.
- Foldseek structure preprocessing.
- Full training or full evaluation pipelines.
- Demo server components.


## Prerequisites

In Google Colab, choose **Runtime > Change runtime type > GPU** if a GPU is available. The notebook also supports CPU fallback for the small toy demo.

Baseline ProTrek-35M files can be downloaded from Hugging Face or provided manually. The fine-tuned ProTrek-HM checkpoint is not stored in Git because `weights/` is gitignored. To run the optional fine-tuned comparison, upload the inference-only checkpoint to Google Drive, a GitHub Release, or Hugging Face, then fill in the URL or Drive path below.

If the fine-tuned checkpoint is missing, the notebook still runs the baseline-only demo.


In [ ]:
# Runtime check
import os
import platform
import sys
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())

try:
    import torch
    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("torch import failed:", exc)


In [ ]:
# Clone the project in Colab, or use the current local checkout.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Dylan-Hzc/ProTrek-HM.git"
BRANCH = "trackb-colabprotrek-hm"
# If this branch has not been pushed yet, replace BRANCH with "main" after merging
# or with the published branch name used for the course submission.

IN_COLAB = "COLAB_GPU" in os.environ or "google.colab" in sys.modules
PROJECT_DIR = Path("/content/ProTrek") if IN_COLAB else Path.cwd()

if IN_COLAB and not PROJECT_DIR.exists():
    subprocess.check_call(["git", "clone", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)])

os.chdir(PROJECT_DIR)
print("cwd:", Path.cwd())


In [ ]:
# Install minimal dependencies. Colab usually already provides a CUDA-compatible torch.
import subprocess
import sys
from pathlib import Path

requirements = Path("requirements_colab.txt")
if requirements.exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)])
else:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers==4.28.0",
        "huggingface_hub",
        "pandas",
        "scikit-learn",
        "matplotlib",
    ])

print("Dependency setup finished. Torch was not installed or replaced by this cell.")


## Weight Setup

Expected baseline directory structure:

```text
weights/ProTrek_35M/
  ProTrek_35M.pt
  esm2_t12_35M_UR50D/
  BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext/
  foldseek_t12_35M/
```

Optional fine-tuned inference-only checkpoint:

```text
weights/ProTrek_35M/protrek_hm_35m_inference_only.pt
```

The optional checkpoint is produced from the training checkpoint by removing optimizer state and keeping a loader-compatible top-level `model` key.


In [ ]:
# Configure and prepare ProTrek-35M weights. This cell never downloads 650M files or FAISS indexes.
from pathlib import Path
import shutil
import urllib.request

MODEL_DIR = Path("weights/ProTrek_35M")
BASELINE_CKPT = MODEL_DIR / "ProTrek_35M.pt"
FINETUNED_CKPT = MODEL_DIR / "protrek_hm_35m_inference_only.pt"

DOWNLOAD_BASELINE_FROM_HF = True
HF_MODEL_REPO = "westlake-repl/ProTrek_35M"

# Fill one of these in if you want the optional fine-tuned comparison.
FINETUNED_CKPT_URL = ""
GOOGLE_DRIVE_FINETUNED_PATH = ""

MODEL_DIR.mkdir(parents=True, exist_ok=True)

required_baseline_paths = [
    BASELINE_CKPT,
    MODEL_DIR / "esm2_t12_35M_UR50D",
    MODEL_DIR / "BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext",
]
baseline_ready = all(path.exists() for path in required_baseline_paths)

if DOWNLOAD_BASELINE_FROM_HF and not baseline_ready:
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id=HF_MODEL_REPO,
        local_dir=str(MODEL_DIR),
        local_dir_use_symlinks=False,
    )

if FINETUNED_CKPT_URL and not FINETUNED_CKPT.exists():
    print("Downloading fine-tuned inference-only checkpoint...")
    urllib.request.urlretrieve(FINETUNED_CKPT_URL, FINETUNED_CKPT)

if GOOGLE_DRIVE_FINETUNED_PATH and not FINETUNED_CKPT.exists():
    source = Path(GOOGLE_DRIVE_FINETUNED_PATH)
    if not source.exists():
        raise FileNotFoundError(f"Google Drive checkpoint path not found: {source}")
    shutil.copy2(source, FINETUNED_CKPT)

baseline_ready = all(path.exists() for path in required_baseline_paths)
print("baseline checkpoint exists:", BASELINE_CKPT.exists())
print("fine-tuned checkpoint exists:", FINETUNED_CKPT.exists())

if not baseline_ready:
    missing = [str(path) for path in required_baseline_paths if not path.exists()]
    raise FileNotFoundError(
        "Baseline ProTrek-35M files are missing. Provide them manually or enable Hugging Face download. "
        f"Missing: {missing}"
    )

if not FINETUNED_CKPT.exists():
    print("Fine-tuned checkpoint is missing; optional comparison will be skipped.")


In [ ]:
# Import the wrapper and resolve local paths. This cell does not load model weights.
import json
from protrek_hm_colab import ColabProTrekHM

wrapper = ColabProTrekHM(
    model_dir=str(MODEL_DIR),
    checkpoint_path=str(BASELINE_CKPT),
    device="auto",
    batch_size=1,
    include_structure_encoder="auto",
    scale_by_temperature=False,
)

print("describe:")
print(json.dumps(wrapper.describe(), indent=2, default=str))
print("resolved paths:")
print(json.dumps(wrapper.resolve_paths(), indent=2, default=str))


In [ ]:
# Load the baseline ProTrek-35M model. This may take seconds to minutes depending on runtime.
wrapper.load()
print("active checkpoint:", wrapper.describe()["active_checkpoint_path"])
print("device:", wrapper.describe()["device"])
print("loaded:", wrapper.describe()["loaded"])


In [ ]:
# Toy examples for pipeline validation only; do not treat these scores as biological conclusions.
toy_sequences = [
    "MKTAYIAKQRQISFVKSHFSRQDILDLWIYHTQGYFPDWQNY",
    "GAVLILKKKGHHEAELKPLAQSHATKHKIPIKYLEFISEAIIH",
]

toy_texts = [
    "DNA-binding protein involved in transcription regulation",
    "Membrane transporter involved in ion transport",
]

print("toy sequence count:", len(toy_sequences))
print("toy text count:", len(toy_texts))


In [ ]:
# Baseline sequence-text similarity.
import pandas as pd

baseline_similarity_matrix = wrapper.similarity_matrix(sequences=toy_sequences, texts=toy_texts)
baseline_pair_scores = wrapper.score_pairs(toy_sequences, toy_texts)

similarity_df = pd.DataFrame(
    baseline_similarity_matrix.numpy(),
    index=[f"sequence_{i+1}" for i in range(len(toy_sequences))],
    columns=[f"text_{i+1}" for i in range(len(toy_texts))],
)
display(similarity_df)
print("baseline pair scores:", [round(float(x), 4) for x in baseline_pair_scores])


In [ ]:
# Optional fine-tuned ProTrek-HM comparison. Uses only the inference-only checkpoint.
import gc
import torch

fine_tuned_ran = False
comparison_df = None
active_wrapper = wrapper

if not FINETUNED_CKPT.exists():
    print("Skipping fine-tuned comparison because the inference-only checkpoint is missing:", FINETUNED_CKPT)
else:
    baseline_pair_scores_saved = baseline_pair_scores.detach().cpu()
    del wrapper
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    finetuned_wrapper = ColabProTrekHM(
        model_dir=str(MODEL_DIR),
        checkpoint_path=str(BASELINE_CKPT),
        finetuned_checkpoint_path=str(FINETUNED_CKPT),
        device="auto",
        batch_size=1,
        include_structure_encoder="auto",
        scale_by_temperature=False,
    )
    finetuned_wrapper.load()
    finetuned_similarity_matrix = finetuned_wrapper.similarity_matrix(sequences=toy_sequences, texts=toy_texts)
    finetuned_pair_scores = finetuned_wrapper.score_pairs(toy_sequences, toy_texts)

    comparison_df = pd.DataFrame({
        "pair": [f"pair_{i+1}" for i in range(len(toy_sequences))],
        "baseline_pair_score": baseline_pair_scores_saved.numpy(),
        "finetuned_pair_score": finetuned_pair_scores.numpy(),
    })
    display(comparison_df)
    active_wrapper = finetuned_wrapper
    fine_tuned_ran = True


In [ ]:
# Optional small hard-negative CSV evaluation. No plots, no full evaluation pipeline.
SAMPLE_CSV = Path("dataset/hard_negatives_test.csv")
NUM_SAMPLES = 5
MAX_SEQ_LEN = 512
required_columns = ["anchor_seq", "anchor_text", "hard_neg_seq"]

if not SAMPLE_CSV.exists():
    print("Skipping mini evaluation; sample CSV not found:", SAMPLE_CSV)
else:
    sample_df = pd.read_csv(SAMPLE_CSV, nrows=NUM_SAMPLES)
    missing = [col for col in required_columns if col not in sample_df.columns]
    if missing:
        raise ValueError(f"Sample CSV is missing required columns: {missing}")

    rows = []
    for idx, row in sample_df.dropna(subset=required_columns).iterrows():
        anchor_seq = str(row["anchor_seq"])[:MAX_SEQ_LEN]
        hard_neg_seq = str(row["hard_neg_seq"])[:MAX_SEQ_LEN]
        anchor_text = str(row["anchor_text"])
        positive = float(active_wrapper.score_pairs([anchor_seq], [anchor_text]).item())
        negative = float(active_wrapper.score_pairs([hard_neg_seq], [anchor_text]).item())
        rows.append({
            "row": int(idx),
            "positive_score": positive,
            "negative_score": negative,
            "correct": positive > negative,
        })

    mini_eval_df = pd.DataFrame(rows)
    display(mini_eval_df)
    if len(mini_eval_df):
        print("mini accuracy:", float(mini_eval_df["correct"].mean()))


In [ ]:
# Save notebook outputs for the report appendix.
OUTPUT_DIR = Path("outputs/colab_demo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

toy_similarity_path = OUTPUT_DIR / "toy_similarity_matrix.csv"
toy_pair_scores_path = OUTPUT_DIR / "toy_pair_scores.csv"
similarity_df.to_csv(toy_similarity_path)
pd.DataFrame({
    "pair": [f"pair_{i+1}" for i in range(len(toy_sequences))],
    "baseline_pair_score": baseline_pair_scores.numpy(),
}).to_csv(toy_pair_scores_path, index=False)

print("saved:", toy_similarity_path)
print("saved:", toy_pair_scores_path)

if fine_tuned_ran and comparison_df is not None:
    comparison_path = OUTPUT_DIR / "baseline_vs_finetuned_pair_scores.csv"
    comparison_df.to_csv(comparison_path, index=False)
    print("saved:", comparison_path)


## Screenshot Checklist

Capture these for the course submission:
- Runtime / GPU check.
- Dependency install.
- Weight setup.
- Wrapper path resolution.
- Baseline model loaded.
- Similarity matrix output.
- Optional fine-tuned comparison.
- Final saved outputs.


## Track B Report Guidance

Suggested report sections:
- **Model selection:** ProTrek.
- **Architecture:** sequence, structure, and text encoders.
- **Adaptation:** `ColabProTrekHM` wrapper.
- **Hard-negative extension:** ProTrek-HM fine-tuning objective.
- **Checkpoint export:** inference-only checkpoint with optimizer removed.
- **Experiments:** toy sequence-text demo and optional mini hard-negative evaluation.
- **Limitations:** 35M by default, fine-tuned checkpoint must be hosted separately, structure/Foldseek path is optional.
